In [19]:
import asyncio
from collections.abc import Callable
from datetime import datetime
from typing import Any

from pydantic import BaseModel, Field, ValidationError

### Task 1

In [3]:
class ToolDefinition(BaseModel):
    name: str
    description: str
    parameters: dict[str, Any] = Field(default_factory=dict)

### Task 2

already build this orchestration loop in mini-project

### Task 3

In [5]:
from typing import Any


def detect_tool_loop(
    tool_name: str,
    arguments: dict[str, Any],
    call_history: list[tuple[str, dict[str, Any]]],
) -> None:
    current_call = (tool_name, arguments)

    # repalce with a more robust check for identical calls
    if current_call in call_history:
        raise RuntimeError(
            f"Stuck loop detected: tool '{tool_name}' "
            f"was called repeatedly with identical arguments."
        )

    call_history.append(current_call)

### Task 4

write in test_excercise.py

### Task 5

In [6]:
def get_available_tools(all_tools: dict, stage: str) -> dict:
    staged_tools = {
        "search": {"search_properties", "check_availability"},
        "property_selected": {
            "search_properties",
            "check_availability",
            "get_property_details",
            "check_cancellation_policy",
        },
        "confirmed": {
            "search_properties",
            "check_availability",
            "get_property_details",
            "check_cancellation_policy",
            "create_booking",
        },
    }

    allowed = staged_tools.get(stage, set())

    return {name: tool for name, tool in all_tools.items() if name in allowed}

### Task 6

write in test_excercise.py

### Task 7

In [ ]:
def log_tool_event(
    trace: list[dict[str, Any]],
    event: str,
    data: dict[str, Any],
) -> None:
    trace.append(
        {
            "timestamp": datetime.now().isoformat(),  # noqa
            "event": event,
            "data": data,
        }
    )


traces = []

log_tool_event(
    traces,
    "request",
    {"user_query": "Find hotels in Mumbai"},
)

log_tool_event(
    traces,
    "tool_call",
    {"tool": "search_properties", "arguments": {"city": "Mumbai"}},
)

log_tool_event(
    traces,
    "tool_result",
    {"tool": "search_properties", "result": ["Hotel A", "Hotel B"]},
)

print(traces)

[{'timestamp': '2026-09-28T15:54:01.067615', 'event': 'request', 'data': {'user_query': 'Find hotels in Mumbai'}}, {'timestamp': '2026-09-28T15:54:01.067659', 'event': 'tool_call', 'data': {'tool': 'search_properties', 'arguments': {'city': 'Mumbai'}}}, {'timestamp': '2026-09-28T15:54:01.067683', 'event': 'tool_result', 'data': {'tool': 'search_properties', 'result': ['Hotel A', 'Hotel B']}}]


### Task 8

In [16]:
async def execute_tool_with_timeout(
    tool: Callable[..., Any],
    args: dict[str, Any],
    timeout: float = 5.0,
):
    try:
        result = await asyncio.wait_for(
            tool(**args),
            timeout=timeout,
        )
        return {"type": "tool_result", "result": result}

    except TimeoutError:
        return {
            "type": "tool_result",
            "error": f"Tool timed out after {timeout}s",
        }

In [17]:
async def slow_tool(query: str):
    await asyncio.sleep(10)
    return "done"


result = await execute_tool_with_timeout(
    slow_tool,
    {"query": "hello"},
    timeout=2,
)

print(result)

{'type': 'tool_result', 'error': 'Tool timed out after 2s'}


### Task 9

In [20]:
def validate_tool_call(schema: type[BaseModel], arguments: dict):
    try:
        validated = schema.model_validate(arguments)
        return {"valid": True, "arguments": validated.model_dump()}

    except ValidationError as e:
        return {
            "valid": False,
            "error": f"Invalid tool arguments: {e.errors()}",
        }


class SearchPropertiesArgs(BaseModel):
    city: str
    guests: int


result = validate_tool_call(
    SearchPropertiesArgs,
    {"city": "Mumbai", "guests": "two"},
)

print(result)

{'valid': False, 'error': "Invalid tool arguments: [{'type': 'int_parsing', 'loc': ('guests',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'two', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}]"}


### Task 10

wrote integration test in mini-project 